# UdaciSense: Baseline Performance - Google Colab Pro

**🚀 GPU-accelerated training with Google Colab Pro**

**Before starting:**
1. Enable GPU: Runtime → Change runtime type → Hardware accelerator → GPU (T4)
2. Ensure your project is uploaded to Google Drive
3. Update the `DRIVE_PROJECT_PATH` below

**Expected training time: ~15-20 minutes with T4 GPU** (vs 12 hours on CPU)

## Step 1: Google Drive Setup

In [ ]:
# Cell 1: Mount Google Drive and navigate to project
from google.colab import drive
import os
import sys

# Mount Google Drive
drive.mount('/content/drive')

# UPDATE THIS PATH to where you uploaded your project
DRIVE_PROJECT_PATH = '/content/drive/MyDrive/udacity-projects/udaci-model-optimization/project/starter_kit'

# Navigate to project directory
os.chdir(DRIVE_PROJECT_PATH)
print(f"✅ Changed to directory: {os.getcwd()}")

# Add project root to Python path for imports
project_root = os.path.abspath('../..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)
print(f"✅ Added to Python path: {project_root}")

# Verify project structure
required_dirs = ['src', 'notebooks', 'models']
missing_dirs = []
for d in required_dirs:
    if os.path.exists(d):
        print(f"✅ Found: {d}/")
    else:
        missing_dirs.append(d)
        print(f"❌ Missing: {d}/")

if missing_dirs:
    print(f"\n⚠️ Please upload these directories to your Google Drive: {missing_dirs}")
else:
    print("\n🎉 All required directories found!")

In [ ]:
# Cell 2: Install requirements in Colab
!pip install -q torch>=2.0.0 torchvision>=0.15.0 
!pip install -q matplotlib seaborn pandas scikit-learn pillow tqdm plotly
!pip install -q thop  # For FLOPs calculation

print("✅ All packages installed!")

## Step 2: Verify GPU Setup

In [ ]:
# Cell 3: Check GPU availability
import torch
import warnings
warnings.filterwarnings('ignore')

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"🚀 GPU Available: {gpu_name}")
    print(f"💾 GPU Memory: {gpu_memory:.1f} GB")
    torch.cuda.empty_cache()
else:
    device = torch.device('cpu')
    print("⚠️ No GPU found. Please enable GPU in Runtime → Change runtime type")

print(f"Device: {device}")

## Step 3: Import Project Modules

In [ ]:
# Cell 4: Import all required libraries
%load_ext autoreload
%autoreload 2

import json
import matplotlib.pyplot as plt
import numpy as np
import random
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset

# Import custom modules
try:
    from src.utils import MAX_ALLOWED_ACCURACY_DROP, TARGET_INFERENCE_SPEEDUP, TARGET_MODEL_COMPRESSION
    from src.utils.data_loader import get_household_loaders, get_input_size, print_dataloader_stats, visualize_batch
    from src.utils.model import MobileNetV3_Household, load_model, print_model_summary, train_model
    from src.utils.evaluation import calculate_confusion_matrix, evaluate_model_metrics
    from src.utils.visualization import plot_confusion_matrix, plot_training_history, plot_weight_distribution
    print("✅ All custom modules imported successfully!")
    
    # Print optimization targets
    print(f"\n🎯 Optimization Targets:")
    print(f"   Max accuracy drop: {MAX_ALLOWED_ACCURACY_DROP*100}%")
    print(f"   Target speedup: {TARGET_INFERENCE_SPEEDUP*100}%")
    print(f"   Target compression: {TARGET_MODEL_COMPRESSION*100}%")
    
except ImportError as e:
    print(f"❌ Import error: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Check that your project is uploaded to Google Drive")
    print("   2. Update DRIVE_PROJECT_PATH in Cell 1")
    print("   3. Ensure src/ directory structure is preserved")

In [ ]:
# Cell 5: Set random seed for reproducibility
def set_deterministic_mode(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)
    
    def seed_worker(worker_id):
        worker_seed = seed + worker_id
        np.random.seed(worker_seed)
        random.seed(worker_seed)
    return seed_worker

seed_worker = set_deterministic_mode(42)
g = torch.Generator()
g.manual_seed(42)

print("✅ Reproducibility mode set (seed=42)")

## Step 4: Setup Directories

In [ ]:
# Cell 6: Create directories for results
model_type = "baseline_mobilenet_colab"
models_dir = f"models/{model_type}"
models_ckp_dir = f"{models_dir}/checkpoints"
results_dir = f"results/{model_type}"

# Create directories
os.makedirs(models_ckp_dir, exist_ok=True)
os.makedirs(results_dir, exist_ok=True)

print(f"📁 Created directories:")
print(f"   Models: {models_dir}")
print(f"   Checkpoints: {models_ckp_dir}")
print(f"   Results: {results_dir}")

# These will be synced back to your Google Drive automatically

## Step 5: Load Dataset (GPU Optimized)

In [ ]:
# Cell 7: Load household objects dataset
# Larger batch size for GPU efficiency
GPU_BATCH_SIZE = 256 if torch.cuda.is_available() else 128
NUM_WORKERS = 2  # Colab works best with 2 workers

print(f"🔄 Loading dataset (batch_size={GPU_BATCH_SIZE}, num_workers={NUM_WORKERS})...")

train_loader, test_loader = get_household_loaders(
    image_size="CIFAR", 
    batch_size=GPU_BATCH_SIZE, 
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

# Get class information
class_names = train_loader.dataset.classes
print(f"\n📊 Dataset loaded with {len(class_names)} classes:")
for i, name in enumerate(class_names):
    print(f"   {i}: {name}")

# Print dataset stats
for name, loader in [('Train', train_loader), ('Test', test_loader)]:
    print(f"\n{name} set statistics:")
    print_dataloader_stats(loader, name.lower())

# Show sample images
print("\n🖼️ Sample images from training set:")
visualize_batch(train_loader, num_images=8)

## Step 6: Initialize and Train Model

In [ ]:
# Cell 8: Initialize MobileNetV3 model
print("🏗️ Initializing MobileNetV3 model...")
model = MobileNetV3_Household().to(device)
print_model_summary(model)

if torch.cuda.is_available():
    print(f"\n💾 GPU Memory after model loading: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

In [ ]:
# Cell 9: Configure training (optimized for GPU)
# Fewer epochs since GPU trains more efficiently
num_epochs = 30
criterion = nn.CrossEntropyLoss()

# Scale learning rate with batch size
base_lr = 0.001 * (GPU_BATCH_SIZE / 128)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=base_lr,
    weight_decay=1e-4,
    betas=(0.9, 0.999)
)

# OneCycleLR for faster convergence
max_lr = 0.005 * (GPU_BATCH_SIZE / 128)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=max_lr,
    steps_per_epoch=len(train_loader),
    epochs=num_epochs,
    pct_start=0.3,
    div_factor=25,
    final_div_factor=1000
)

training_config = {
    'num_epochs': num_epochs,
    'criterion': criterion,
    'optimizer': optimizer,
    'scheduler': scheduler,
    'patience': 7,
    'device': device
}

print(f"⚙️ Training Configuration:")
print(f"   Epochs: {num_epochs}")
print(f"   Base LR: {base_lr:.6f}")
print(f"   Max LR: {max_lr:.6f}")
print(f"   Batch Size: {GPU_BATCH_SIZE}")
print(f"   Device: {device}")

In [ ]:
# Cell 10: Train the model
checkpoint_path = f"{models_ckp_dir}/model.pth"

print(f"🚀 Starting training on {device}")
if torch.cuda.is_available():
    print(f"   Expected time: ~15-20 minutes on T4 GPU")
else:
    print(f"   ⚠️ Running on CPU - this will take much longer!")

start_time = time.time()

# Train the model
training_stats, best_accuracy, best_epoch = train_model(
    model,
    train_loader,
    test_loader,
    training_config,
    checkpoint_path=checkpoint_path,
)

training_time = time.time() - start_time
print(f"\n✅ Training completed in {training_time/60:.1f} minutes!")
print(f"🏆 Best accuracy: {best_accuracy:.4f} at epoch {best_epoch}")

# Save training stats
with open(f"{results_dir}/training_stats.json", 'w') as f:
    json.dump(training_stats, f, indent=4)

if torch.cuda.is_available():
    cpu_time_hours = 12  # Estimated CPU time
    speedup = (cpu_time_hours * 60) / (training_time / 60)
    print(f"\n🚀 GPU Speedup: ~{speedup:.1f}x faster than CPU training!")

## Step 7: Evaluate Model Performance

In [ ]:
# Cell 11: Comprehensive model evaluation
print("🔍 Evaluating model performance...")

# Load best model
model = load_model(checkpoint_path, device)

# Evaluation parameters
n_classes = len(class_names)
input_size = get_input_size("CIFAR")

# Calculate metrics
start_eval = time.time()
baseline_metrics = evaluate_model_metrics(
    model, test_loader, device, n_classes, class_names, input_size,
    save_path=f"{results_dir}/metrics.json"
)
eval_time = time.time() - start_eval

print(f"✅ Evaluation completed in {eval_time:.1f} seconds")

# Display key metrics
print(f"\n📊 BASELINE PERFORMANCE:")
print(f"   🎯 Top-1 Accuracy: {baseline_metrics['accuracy']['top1_acc']:.2f}%")
print(f"   🎯 Top-5 Accuracy: {baseline_metrics['accuracy']['top5_acc']:.2f}%")
print(f"   📏 Model Size: {baseline_metrics['size']['model_size_mb']:.2f} MB")
print(f"   🔢 Parameters: {baseline_metrics['size']['num_params']:,}")
print(f"   ⏱️ CPU Inference: {baseline_metrics['timing']['cpu']['avg_time_ms']:.2f} ms")
if torch.cuda.is_available():
    print(f"   ⚡ GPU Inference: {baseline_metrics['timing']['cuda']['avg_time_ms']:.2f} ms")

In [ ]:
# Cell 12: Generate visualizations
print("📈 Generating visualizations...")

# Confusion matrix
confusion_matrix = calculate_confusion_matrix(model, test_loader, device, n_classes)
plot_confusion_matrix(confusion_matrix, class_names, f"{results_dir}/confusion_matrix.png")

# Training history
plot_training_history(training_stats, f"{results_dir}/training_history.png")

# Weight distribution
plot_weight_distribution(model, output_path=f"{results_dir}/weight_distribution.png")

print("✅ All visualizations saved to results directory")

## Step 8: Calculate Optimization Targets

In [ ]:
# Cell 13: Calculate and display optimization targets
# Calculate targets based on CTO requirements
target_model_size = baseline_metrics['size']['model_size_mb'] * (1 - TARGET_MODEL_COMPRESSION)
target_cpu_time = baseline_metrics['timing']['cpu']['avg_time_ms'] * (1 - TARGET_INFERENCE_SPEEDUP)
min_accuracy = baseline_metrics['accuracy']['top1_acc'] * (1 - MAX_ALLOWED_ACCURACY_DROP)

print("\n" + "="*60)
print("🎯 OPTIMIZATION TARGETS FOR COMPRESSION")
print("="*60)

print(f"\n📏 SIZE TARGET:")
print(f"   Current: {baseline_metrics['size']['model_size_mb']:.2f} MB")
print(f"   Target:  {target_model_size:.2f} MB")
print(f"   Reduction: {TARGET_MODEL_COMPRESSION*100}%")

print(f"\n⏱️ SPEED TARGET (CPU):")
print(f"   Current: {baseline_metrics['timing']['cpu']['avg_time_ms']:.2f} ms")
print(f"   Target:  {target_cpu_time:.2f} ms")
print(f"   Speedup: {TARGET_INFERENCE_SPEEDUP*100}%")

if torch.cuda.is_available():
    target_gpu_time = baseline_metrics['timing']['cuda']['avg_time_ms'] * (1 - TARGET_INFERENCE_SPEEDUP)
    print(f"\n⚡ SPEED TARGET (GPU):")
    print(f"   Current: {baseline_metrics['timing']['cuda']['avg_time_ms']:.2f} ms")
    print(f"   Target:  {target_gpu_time:.2f} ms")
    print(f"   Speedup: {TARGET_INFERENCE_SPEEDUP*100}%")

print(f"\n🎯 ACCURACY TARGET:")
print(f"   Current: {baseline_metrics['accuracy']['top1_acc']:.2f}%")
print(f"   Minimum: {min_accuracy:.2f}%")
print(f"   Max drop: {MAX_ALLOWED_ACCURACY_DROP*100}%")

print("\n" + "="*60)

# Save targets for next notebook
targets = {
    'baseline_accuracy': baseline_metrics['accuracy']['top1_acc'],
    'baseline_size_mb': baseline_metrics['size']['model_size_mb'],
    'baseline_cpu_ms': baseline_metrics['timing']['cpu']['avg_time_ms'],
    'target_size_mb': target_model_size,
    'target_cpu_ms': target_cpu_time,
    'min_accuracy': min_accuracy,
    'training_time_minutes': training_time / 60
}

if torch.cuda.is_available():
    targets['baseline_gpu_ms'] = baseline_metrics['timing']['cuda']['avg_time_ms']
    targets['target_gpu_ms'] = target_gpu_time

with open(f"{results_dir}/optimization_targets.json", 'w') as f:
    json.dump(targets, f, indent=4)

print(f"💾 Results saved to: {results_dir}/")
print(f"📁 All files are automatically synced to your Google Drive!")

## 🎉 Training Complete!

**Your baseline model has been successfully trained with GPU acceleration!**

### 📊 What You Achieved:
- ✅ **GPU Training**: ~15-20 minutes vs 12 hours on CPU
- ✅ **Baseline Established**: Ready for optimization
- ✅ **All Results Saved**: Models, metrics, and visualizations in Google Drive

### 🚀 Next Steps:
1. **Review your results** in the `results/baseline_mobilenet_colab/` folder
2. **Implement compression techniques** in the next notebook
3. **Meet the CTO targets**: 70% size reduction, 60% speed improvement

### 💾 Your Files (Auto-synced to Drive):
- `models/baseline_mobilenet_colab/checkpoints/model.pth` - Best model
- `results/baseline_mobilenet_colab/metrics.json` - Performance metrics
- `results/baseline_mobilenet_colab/*.png` - Visualizations

**Ready to optimize! 🎯**